In [1]:
import pandas as pd
import ipywidgets as w
from IPython.display import display, clear_output

# In-memory storage: list of dicts
students = []  # each: {"id": str, "name": str, "program": str, "gpa": float | None}

# Widgets
id_inp      = w.Text(description="ID", placeholder="e.g., 101", layout=w.Layout(width="220px"))
name_inp    = w.Text(description="Name", placeholder="e.g., Alex", layout=w.Layout(width="220px"))
prog_inp    = w.Text(description="Program", placeholder="e.g., CS", layout=w.Layout(width="220px"))
grades_inp  = w.Text(description="Grades", placeholder="e.g., 90, 85, 78", layout=w.Layout(width="220px"))

add_btn     = w.Button(description="Add/Update", button_style="success")
del_btn     = w.Button(description="Delete by ID", button_style="danger")
search_inp  = w.Text(description="Search", placeholder="ID or Name", layout=w.Layout(width="220px"))
search_btn  = w.Button(description="Search", button_style="info")
show_all_btn= w.Button(description="Show All", button_style="primary")

msg_out  = w.Output()
table_out = w.Output()

def compute_gpa(grades_text: str):
    if not grades_text.strip():
        return None
    try:
        nums = [float(x) for x in grades_text.replace(";", ",").split(",") if x.strip()]
        return round(sum(nums) / len(nums), 2) if nums else None
    except ValueError:
        return "error"

def as_df(data):
    if not data:
        return pd.DataFrame(columns=["ID", "Name", "Program", "GPA"])
    return pd.DataFrame(data).rename(columns={"id": "ID", "name": "Name", "program": "Program", "gpa": "GPA"})

def refresh_table(rows=None):
    with table_out:
        clear_output()
        df = as_df(students if rows is None else rows)
        display(df.style.hide(axis='index'))

def set_message(text, level="info"):
    colors = {"info": "blue", "error": "red", "success": "green"}
    with msg_out:
        clear_output()
        print(f"\x1b[1;{34 if level=='info' else 31 if level=='error' else 32}m{text}\x1b[0m")

def add_or_update(_):
    sid = id_inp.value.strip()
    name = name_inp.value.strip()
    prog = prog_inp.value.strip()
    gpa = compute_gpa(grades_inp.value)
    if gpa == "error":
        set_message("Invalid grades. Use comma-separated numbers (e.g., 90,85,78).", "error")
        return
    if not sid or not name:
        set_message("ID and Name are required.", "error")
        return
    existing = next((s for s in students if s["id"] == sid), None)
    if existing:
        existing.update({"name": name, "program": prog, "gpa": gpa})
        set_message(f"Updated student ID {sid}.", "success")
    else:
        students.append({"id": sid, "name": name, "program": prog, "gpa": gpa})
        set_message(f"Added student ID {sid}.", "success")
    refresh_table()

def delete_student(_):
    sid = id_inp.value.strip()
    if not sid:
        set_message("Enter an ID to delete.", "error")
        return
    before = len(students)
    students[:] = [s for s in students if s["id"] != sid]
    if len(students) < before:
        set_message(f"Deleted student ID {sid}.", "success")
    else:
        set_message(f"No student found with ID {sid}.", "error")
    refresh_table()

def search_student(_):
    term = search_inp.value.strip().lower()
    if not term:
        set_message("Enter an ID or Name to search.", "error")
        return
    results = [s for s in students if term in s["id"].lower() or term in s["name"].lower()]
    if results:
        set_message(f"Found {len(results)} result(s).", "success")
        refresh_table(results)
    else:
        set_message("No matches found.", "error")
        refresh_table([])

def show_all(_):
    refresh_table()
    set_message(f"Total students: {len(students)}", "info")

add_btn.on_click(add_or_update)
del_btn.on_click(delete_student)
search_btn.on_click(search_student)
show_all_btn.on_click(show_all)

# Layout
form = w.VBox([
    w.HBox([id_inp, name_inp]),
    w.HBox([prog_inp, grades_inp]),
    w.HBox([add_btn, del_btn, search_inp, search_btn, show_all_btn]),
    msg_out,
    table_out
])

display(form)
refresh_table([])
set_message("Ready. Add students, then search/delete/show.", "info")